# 📈 Pandas pour la finance — analyser des cours de bourse

## Le contexte

Tu rejoins l'équipe d'un gérant de portefeuille. Ta mission : comparer quatre actions cotées à New York et à Paris de janvier 2023 à septembre 2026, mesurer leur performance et leur risque, puis tester une stratégie d'investissement simple.

| Ticker | Entreprise | Place de cotation | Devise |
|---|---|---|---|
| `AAPL` | Apple | New York (Nasdaq) | dollar |
| `MSFT` | Microsoft | New York (Nasdaq) | dollar |
| `MC.PA` | LVMH | Paris (Euronext) | euro |
| `TTE.PA` | TotalEnergies | Paris (Euronext) | euro |

On récupère aussi le taux de change `EURUSD=X` pour pouvoir tout comparer en euros.

## Les données

Les cours viennent de **Yahoo Finance** via la bibliothèque `yfinance`. Pour chaque action et chaque séance de bourse :

| Colonne | Signification |
|---|---|
| `Open` | cours à l'ouverture |
| `High` / `Low` | plus haut et plus bas de la séance |
| `Close` | cours à la clôture, **ajusté** des dividendes et des divisions d'actions |
| `Volume` | nombre d'actions échangées |

## Le programme

| Partie | Thème | Méthodes |
|---|---|---|
| 1 | La structure des données yfinance | `MultiIndex`, `xs` |
| 2 | Sélectionner | `loc`, `iloc`, dates |
| 3 | Les trous du calendrier | `isna`, `ffill` |
| 4 | Les rendements | `shift`, `diff`, `pct_change`, `idxmax`, `nlargest` |
| 5 | Performance et pertes | `cumprod`, `cummax`, `expanding` |
| 6 | Les fenêtres glissantes | `rolling`, `ewm` |
| 7 | Changer de fréquence | `resample` |
| 8 | Empiler des tableaux | `concat` |
| 9 | Joindre des tableaux | `merge`, `merge_asof` |
| 10 | Calculs par action | `pivot`, `melt`, `groupby` + `transform` / `shift` / `rank` |
| 11 | Découper en classes | `qcut`, `cut` |
| 12 | Les corrélations | `corr` |
| 13 | Projet : tester une stratégie | `np.where`, `shift`, tout le reste |
| Bonus | Écrire en chaîne | `assign`, `pipe` |

⚠️ Les parties s'enchaînent : les variables créées dans une partie servent dans les suivantes. Fais les exercices dans l'ordre.

> Ce notebook est un exercice de programmation, pas un conseil en investissement.

---
# Partie 0 — Mise en place

Si ce n'est pas déjà fait, installe les bibliothèques (une seule fois) :

```
pip install pandas plotly yfinance
```

La cellule suivante est **déjà écrite** : exécute-la, puis lis-la. Elle télécharge toutes les données en un seul appel à `yf.download`, puis sépare les actions du taux de change.

**À propos de yfinance** : ce n'est pas une API officielle. La bibliothèque interroge le site de Yahoo Finance, qui peut limiter les requêtes ou changer de format. Si le téléchargement échoue (pas de connexion, trop de requêtes…), la cellule charge à la place le fichier `cours_bourse_simules.csv`, qui doit se trouver dans le même dossier que le notebook. Ces données de secours ont **exactement le même format** et suivent les vrais calendriers de cotation, mais **les prix sont simulés** : tous les exercices fonctionnent, mais les résultats ne disent rien des vraies entreprises. Le message affiché t'indique quelle source est utilisée.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ACTIONS = ["AAPL", "MSFT", "MC.PA", "TTE.PA"]   # Apple, Microsoft, LVMH, TotalEnergies
CHANGE = "EURUSD=X"                              # taux de change : combien de dollars vaut 1 euro
DEBUT, FIN = "2023-01-01", "2026-10-01"          # dates fixes : tout le monde travaille sur la même période
FICHIER_SECOURS = "cours_bourse_simules.csv"
COLONNES = ["Close", "High", "Low", "Open", "Volume"]


def charger_cours(tickers):
    """Télécharge les cours avec yfinance ; en cas d'échec, charge les données simulées de secours."""
    try:
        import logging
        import yfinance as yf

        logging.getLogger("yfinance").setLevel(logging.CRITICAL)   # on affiche notre propre message
        donnees = yf.download(tickers, start=DEBUT, end=FIN, auto_adjust=True, progress=False)
        manquants = [t for t in tickers if donnees.empty or donnees[("Close", t)].dropna().empty]
        if manquants:
            raise ValueError(f"aucune donnée reçue pour {manquants}")
        print("✅ Données réelles téléchargées depuis Yahoo Finance")
        return donnees[COLONNES]
    except Exception as erreur:
        print(f"⚠️ Téléchargement impossible : {erreur}")
        print(f"   → Utilisation des données SIMULÉES de « {FICHIER_SECOURS} » : les prix sont fictifs.")
        secours = pd.read_csv(FICHIER_SECOURS, parse_dates=["Date"])
        secours = secours[secours["Ticker"].isin(tickers)]
        donnees = secours.pivot(index="Date", columns="Ticker", values=COLONNES)
        donnees.columns.names = ["Price", "Ticker"]
        return donnees


toutes = charger_cours(ACTIONS + [CHANGE])

# On sépare les actions et le taux de change
brut = toutes.drop(columns=CHANGE, level="Ticker").dropna(how="all")
brut.columns = brut.columns.remove_unused_levels()
fx_brut = toutes.xs(CHANGE, axis=1, level="Ticker").dropna(how="all")

print("Actions :", brut.shape, "| Taux de change :", fx_brut.shape)

---
# Partie 1 — La structure des données yfinance

Quand on télécharge plusieurs actions d'un coup, yfinance renvoie un tableau **large** dont les colonnes ont **deux niveaux** (un `MultiIndex`) :

```
Price     Close                   High            …   Volume
Ticker    AAPL  MC.PA  MSFT  TTE.PA  AAPL  MC.PA …    AAPL …
Date
```

- niveau `Price` : le type de valeur (`Close`, `High`, `Low`, `Open`, `Volume`) ;
- niveau `Ticker` : l'action.

L'index des lignes est la **date** de la séance (un `DatetimeIndex`).

**Aide-mémoire**

| Besoin | Code |
|---|---|
| Un niveau entier | `brut["Close"]` → un tableau avec une colonne par action |
| Une colonne précise | `brut[("Close", "AAPL")]` |
| Toutes les colonnes d'une action | `brut.xs("AAPL", axis=1, level="Ticker")` |

### Exercice 1.1 — Premier coup d'œil

Affiche les dimensions de `brut`, les noms des niveaux de colonnes (`brut.columns.names`), puis les 5 premières lignes.

In [ ]:
# Ton code ici

### Exercice 1.2 — Les cours de clôture

Crée `cloture`, le tableau des cours de clôture (une colonne par action), et affiche ses 5 dernières lignes.

C'est le tableau dont on se servira le plus.

In [ ]:
# Ton code ici

### Exercice 1.3 — Une seule colonne, deux syntaxes

Récupère le volume de `MSFT` de deux façons : en sélectionnant les niveaux l'un après l'autre, puis avec un tuple. Vérifie avec `.equals()` que le résultat est identique.

In [ ]:
# Ton code ici

### Exercice 1.4 — Toutes les colonnes d'une action avec `xs`

Crée `aapl`, le tableau des cinq colonnes d'Apple (`Open`, `High`, `Low`, `Close`, `Volume`), en supprimant les lignes vides avec `dropna()`.

Pourquoi y a-t-il des lignes vides pour Apple ? Indice : le tableau `brut` contient les séances de New York **et** de Paris.

In [ ]:
# Ton code ici

---
# Partie 2 — Sélectionner avec `loc` et `iloc`

| | `iloc` | `loc` |
|---|---|---|
| Sélectionne par | **position** (0, 1, 2…) | **étiquette** (une date, un nom de colonne) |
| Borne de fin d'une tranche | **exclue** | **incluse** |
| Exemple | `df.iloc[-5:, :2]` | `df.loc["2024-03-15", "Close"]` |

Avec un index de dates, `loc` accepte des **dates partielles** : `df.loc["2024"]` (toute l'année), `df.loc["2024-03"]` (tout le mois), `df.loc["2024-03":"2024-06"]` (de mars à juin inclus).

Les dates de l'index ont aussi des attributs : `df.index.year`, `df.index.month`, `df.index.day_name()`.

### Exercice 2.1 — `iloc`

Dans `aapl`, affiche : la première ligne, la dernière ligne, puis les 5 dernières lignes limitées aux 2 premières colonnes.

In [ ]:
# Ton code ici

### Exercice 2.2 — `loc` avec des dates

Dans `aapl` :

1. affiche le cours de clôture du 15 mars 2024 ;
2. compte le nombre de séances en 2024 ;
3. affiche les cours de clôture de mars et avril 2025.

In [ ]:
# Ton code ici

### Exercice 2.3 — `loc` avec un masque

Trouve les séances où le volume d'Apple a dépassé **deux fois son volume médian**. Affiche uniquement les colonnes `Close` et `Volume`, et compte ces séances.

In [ ]:
# Ton code ici

### Exercice 2.4 — Les attributs de date

1. Combien de séances Apple compte-t-elle chaque année ? (`aapl.index.year`)
2. Combien de séances par jour de la semaine ? Y a-t-il des séances le samedi ?

In [ ]:
# Ton code ici

### Exercice 2.5 — Le piège des bornes

Récupère les 3 premières séances d'Apple avec `iloc`, puis avec `loc` en utilisant les dates de la première et de la troisième séance.

Observe : pour obtenir 3 lignes, quelle borne de fin faut-il donner dans chaque cas ?

In [ ]:
# Ton code ici

---
# Partie 3 — Les trous du calendrier

New York et Paris n'ont pas les mêmes jours fériés : le 4 juillet, la Bourse de New York est fermée et celle de Paris ouverte ; le 1er mai, c'est l'inverse. Dans le tableau `cloture`, ces jours-là apparaissent comme des **valeurs manquantes**.

La convention en finance : un jour de fermeture, le prix d'une action est **son dernier cours connu**. C'est exactement ce que fait `ffill()` (*forward fill*) : il recopie la dernière valeur disponible vers l'avant.

⚠️ On ne remplit **jamais** avec la valeur suivante (`bfill()`) : ce serait utiliser un prix qu'on ne connaissait pas encore ce jour-là.

### Exercice 3.1 — Où sont les trous ?

1. Compte les valeurs manquantes de chaque colonne de `cloture`.
2. Affiche les 10 premières lignes qui contiennent au moins une valeur manquante. Reconnais-tu des dates ?

💡 `cloture.isna().any(axis=1)` vaut `True` pour les lignes qui ont au moins un trou.

In [ ]:
# Ton code ici

### Exercice 3.2 — Combler avec `ffill`

Crée `prix` : le tableau `cloture` dont les trous sont comblés avec `ffill()`. Reste-t-il des valeurs manquantes ? Où, et pourquoi ?

Supprime-les avec `dropna()`, puis vérifie qu'il n'y a plus aucun trou.

In [ ]:
# Ton code ici

---
# Partie 4 — Les rendements : `shift`, `diff`, `pct_change`

Le **rendement** d'une journée, c'est la variation du prix en pourcentage par rapport à la veille :

> rendement = cours du jour / cours de la veille − 1

En finance, on raisonne en rendements plutôt qu'en prix : une hausse de 5 € n'a pas le même sens pour une action à 50 € et pour une action à 700 €.

**Aide-mémoire**

| Méthode | Effet |
|---|---|
| `s.shift(1)` | décale les valeurs d'une ligne vers le bas : chaque ligne reçoit la valeur de la **veille** |
| `s.shift(-1)` | décale vers le haut : chaque ligne reçoit la valeur du **lendemain** |
| `s.diff()` | différence avec la ligne précédente (`s - s.shift(1)`) |
| `s.pct_change()` | variation en pourcentage (`s / s.shift(1) - 1`) |

### Exercice 4.1 — Le rendement à la main avec `shift`

Dans `aapl`, crée :

- `cloture_veille` : le cours de clôture de la séance précédente ;
- `rendement_main` : le rendement du jour, calculé avec la formule ci-dessus.

Affiche les colonnes `Close`, `cloture_veille` et `rendement_main` des 3 premières lignes. Pourquoi la première ligne est-elle vide ?

In [ ]:
# Ton code ici

### Exercice 4.2 — `pct_change` et `diff`

Crée dans `aapl` :

- `rendement` avec `pct_change()`, et vérifie qu'il donne le même résultat que `rendement_main` (`np.allclose` compare des nombres à virgule en tolérant les arrondis) ;
- `variation` avec `diff()` : la variation du cours en dollars.

In [ ]:
# Ton code ici

### Exercice 4.3 — Regarder le lendemain avec `shift(-1)`

Crée `rendement_lendemain` : le rendement de la séance **suivante**. Affiche les 3 dernières lignes de `rendement` et `rendement_lendemain`.

Cette colonne est utile pour **analyser** le passé (« que se passe-t-il le lendemain d'une forte baisse ? »), mais elle contient une information qu'on ne connaît pas le jour même. Il ne faut jamais s'en servir pour prendre une décision ce jour-là : on y reviendra dans le projet final.

In [ ]:
# Ton code ici

### Exercice 4.4 — Les rendements de toutes les actions

1. Crée `rendements`, les rendements journaliers des quatre actions à partir de `prix`, sans la première ligne vide.
2. Affiche leurs statistiques avec `describe()`.
3. Pour chaque action, trouve la date de la meilleure et de la pire séance (`idxmax`, `idxmin`).
4. Affiche les 5 meilleures séances d'Apple (`nlargest`).

In [ ]:
# Ton code ici

### Exercice 4.5 — La volatilité

La **volatilité** mesure l'ampleur des variations : c'est l'écart-type des rendements. On l'exprime sur un an en multipliant l'écart-type journalier par la racine carrée de 252 (le nombre de séances dans une année).

Calcule la volatilité annualisée de chaque action, en pourcentage, de la plus risquée à la moins risquée.

In [ ]:
# Ton code ici

---
# Partie 5 — Performance et pertes : `cumprod`, `cummax`, `expanding`

Les fonctions **cumulées** calculent, à chaque date, un résultat sur tout ce qui précède :

| Méthode | À chaque ligne |
|---|---|
| `cumsum()` | somme depuis le début |
| `cumprod()` | produit depuis le début |
| `cummax()` / `cummin()` | plus grande / plus petite valeur vue jusque-là |
| `expanding().mean()` (ou `.std()`, `.max()`…) | n'importe quel calcul sur toutes les lignes depuis le début |

Deux usages incontournables en finance :

- la **performance cumulée** : ce que devient un placement de départ. Si un placement fait +10 % puis −10 %, il vaut 1,10 × 0,90 = 0,99 : on **multiplie** les (1 + rendement), d'où `cumprod` ;
- le **drawdown** : la chute par rapport au plus haut atteint jusque-là, d'où `cummax`.

### Exercice 5.1 — La base 100

Pour comparer des actions qui ne cotent pas au même prix, on les ramène toutes à 100 au premier jour.

1. Crée `base100` : chaque colonne de `prix` divisée par sa première valeur, multipliée par 100.
2. Vérifie que la dernière ligne est identique à `100 * (1 + rendements).cumprod()` : les deux méthodes sont équivalentes.
3. Trace `base100` avec `px.line`.

💡 `px.line` accepte directement un tableau large : chaque colonne devient une courbe.

In [ ]:
# Ton code ici

### Exercice 5.2 — Le drawdown

Le drawdown répond à la question : « si j'avais acheté au pire moment, au plus haut, combien aurais-je perdu ? »

> drawdown = valeur du jour / plus haut atteint jusque-là − 1

1. Crée `drawdown` à partir de `base100`.
2. Pour chaque action, affiche la pire baisse (en %) et la date à laquelle elle a été atteinte.
3. Trace le drawdown d'Apple avec `px.area`.

In [ ]:
# Ton code ici

### Exercice 5.3 — `expanding`

1. Vérifie que `base100.expanding().max()` donne exactement la même chose que `base100.cummax()`.
2. Calcule le rendement journalier moyen d'Apple « depuis le début » avec `expanding().mean()`, et affiche-le tous les 6 mois (`.iloc[::126]`). Que remarques-tu sur sa stabilité au fil du temps ?

In [ ]:
# Ton code ici

---
# Partie 6 — Les fenêtres glissantes : `rolling` et `ewm`

`rolling(n)` calcule une statistique sur les **n dernières lignes**, et fait glisser cette fenêtre jour après jour. C'est la base de la plupart des indicateurs techniques.

```python
s.rolling(20).mean()     # moyenne mobile sur 20 séances (environ un mois de bourse)
s.rolling(20).std()      # écart-type glissant
s.rolling(252).max()     # plus haut sur un an
```

Les n − 1 premières lignes sont vides : la fenêtre n'est pas encore pleine. Le paramètre `min_periods` permet de calculer malgré tout avec moins de lignes.

`ewm(span=n)` calcule une moyenne **exponentielle** : au lieu de donner le même poids aux n jours, elle donne plus de poids aux jours récents. Elle réagit donc plus vite.

| Famille | Fenêtre | Poids |
|---|---|---|
| `rolling(n)` | les n dernières lignes | identiques |
| `expanding()` | tout depuis le début | identiques |
| `ewm(span=n)` | tout depuis le début | décroissants avec l'ancienneté |

### Exercice 6.1 — Les moyennes mobiles

Dans `aapl`, crée `mm_20` et `mm_50`, les moyennes mobiles du cours de clôture sur 20 et 50 séances.

Combien de valeurs manquantes contient `mm_50`, et pourquoi ce nombre ? Calcule ensuite la même moyenne avec `min_periods=1` et compare les premières lignes.

In [ ]:
# Ton code ici

### Exercice 6.2 — Le graphique des moyennes mobiles

Trace le cours d'Apple avec ses deux moyennes mobiles, depuis octobre 2025.

💡 `px.line(tableau, y=["col1", "col2", "col3"])` trace plusieurs colonnes d'un coup.

In [ ]:
# Ton code ici

### Exercice 6.3 — La volatilité glissante

Calcule la volatilité annualisée **sur 20 séances glissantes** des quatre actions (en %), et trace-la. Le risque est-il constant dans le temps ?

In [ ]:
# Ton code ici

### Exercice 6.4 — Le plus haut sur un an

Dans `aapl`, crée `plus_haut_1an` : le plus haut cours de clôture sur les 252 dernières séances. Puis `ecart_plus_haut` : l'écart en % entre le cours du jour et ce plus haut.

À combien du plus haut annuel se trouve Apple à la dernière séance ?

In [ ]:
# Ton code ici

### Exercice 6.5 — La moyenne exponentielle

Dans `aapl`, crée `mme_20` : la moyenne exponentielle sur 20 séances (`ewm(span=20, adjust=False).mean()`).

Trace `Close`, `mm_20` et `mme_20` sur les 120 dernières séances : laquelle des deux moyennes suit le cours de plus près ?

In [ ]:
# Ton code ici

### Exercice 6.6 — La corrélation glissante

Calcule la corrélation entre les rendements d'Apple et de Microsoft sur 60 séances glissantes, et trace-la. Le lien entre les deux actions est-il stable ?

💡 `serie_a.rolling(60).corr(serie_b)`

In [ ]:
# Ton code ici

---
# Partie 7 — Changer de fréquence avec `resample`

`resample` est le `groupby` des dates : il regroupe les lignes par période (semaine, mois, année) puis applique un calcul. L'index doit être un `DatetimeIndex`.

| Code | Période |
|---|---|
| `"W"` | semaine |
| `"ME"` | mois (étiqueté au dernier jour du mois) |
| `"QE"` | trimestre |
| `"YE"` | année |

```python
prix.resample("ME").last()                       # dernier cours de chaque mois
aapl.resample("W")["Volume"].sum()               # volume total de chaque semaine
aapl.resample("ME").agg({"Open": "first", "Close": "last"})   # un calcul différent par colonne
```

⚠️ Les anciens codes `"M"` et `"Y"` sont obsolètes : utilise `"ME"` et `"YE"`.

### Exercice 7.1 — Les rendements mensuels

Crée `prix_mensuel` (le dernier cours de chaque mois, à partir de `prix`), puis `rendements_mensuels` avec `pct_change()`. Affiche les 6 premières lignes de chacun, arrondies.

In [ ]:
# Ton code ici

### Exercice 7.2 — Les bougies mensuelles

Une **bougie** résume une période avec 4 prix : l'ouverture (le premier `Open`), le plus haut (le max des `High`), le plus bas (le min des `Low`) et la clôture (le dernier `Close`).

1. Crée `bougies` : les bougies mensuelles d'Apple, avec en plus le volume total du mois.
2. Trace-les avec `go.Candlestick` (code du graphique fourni dans l'aide ci-dessous).

```python
fig = go.Figure(go.Candlestick(x=bougies.index, open=bougies["Open"], high=bougies["High"],
                               low=bougies["Low"], close=bougies["Close"]))
fig.update_layout(title="…", xaxis_rangeslider_visible=False)
fig.show()
```

In [ ]:
# Ton code ici

### Exercice 7.3 — La performance par année civile

Calcule la performance de chaque action pour chaque année, en %, en composant les rendements journaliers de l'année : produit des (1 + rendement), moins 1.

💡 `resample` accepte n'importe quelle agrégation : `(1 + rendements).resample("YE").prod()`.

Trace le résultat en barres groupées. Pense à indiquer que 2026 est une année incomplète.

In [ ]:
# Ton code ici

---
# Partie 8 — Empiler des tableaux avec `concat`

`pd.concat` colle des tableaux les uns aux autres :

- `axis=0` (par défaut) : **les uns sous les autres** (on ajoute des lignes) ;
- `axis=1` : **côte à côte** (on ajoute des colonnes), en alignant les lignes sur l'index.

```python
pd.concat([df1, df2])                          # empiler
pd.concat([df1, df2], ignore_index=True)       # empiler et renuméroter les lignes
pd.concat({"A": df1, "B": df2})                # empiler en gardant l'origine dans un niveau d'index
pd.concat([s1, s2], axis=1)                    # côte à côte, alignement sur l'index (jointure externe)
pd.concat([s1, s2], axis=1, join="inner")      # côte à côte, seulement les index communs
```

`concat` **n'utilise pas de clé** : il empile ou aligne sur l'index. Pour associer des lignes selon la valeur d'une colonne, c'est `merge` (partie 9).

### Exercice 8.1 — Passer au format long

Le format large (une colonne par action) est pratique pour les calculs vectoriels, mais beaucoup d'outils (`groupby`, Plotly, les bases de données) préfèrent le **format long** : une ligne par couple (date, action).

Construis `long` :

1. pour chaque ticker de `ACTIONS`, extrais ses colonnes avec `xs`, supprime les lignes vides et ajoute une colonne `Ticker` ;
2. empile ces quatre tableaux avec `pd.concat` ;
3. remets la date en colonne avec `reset_index()`.

Vérifie le nombre de lignes par action.

In [ ]:
# Ton code ici

### Exercice 8.2 — Côte à côte : l'alignement sur l'index

Récupère les cours de clôture d'Apple et de TotalEnergies **sans les trous** (`cloture["AAPL"].dropna()`…), puis colle-les côte à côte avec `concat(axis=1)` :

1. d'abord avec la jointure par défaut : combien de lignes, combien de trous ?
2. puis avec `join="inner"` : combien de lignes ?

Explique la différence.

In [ ]:
# Ton code ici

### Exercice 8.3 — Empiler deux périodes qui se chevauchent

Situation fréquente : tu as un historique, puis tu télécharges une mise à jour qui recouvre en partie les mêmes dates.

1. Crée `historique = prix.loc[:"2024-12-31"]` et `mise_a_jour = prix.loc["2024-12-01":]`.
2. Empile-les avec `concat`. Combien de dates apparaissent en double ? (`index.duplicated()`)
3. Supprime les doublons en gardant la version la plus récente (`keep="last"`), puis vérifie que tu retrouves exactement `prix`.

In [ ]:
# Ton code ici

---
# Partie 9 — Joindre des tableaux avec `merge`

`merge` associe les lignes de deux tableaux **selon la valeur d'une ou plusieurs colonnes** (la clé), comme une jointure SQL.

```python
pd.merge(gauche, droite, on="Ticker", how="left")
```

| `how` | Lignes conservées |
|---|---|
| `"inner"` (défaut) | seulement les clés présentes **des deux côtés** |
| `"left"` | toutes les lignes de gauche ; trous si pas de correspondance à droite |
| `"right"` | toutes les lignes de droite |
| `"outer"` | toutes les lignes des deux côtés |

Options utiles : `indicator=True` ajoute une colonne `_merge` qui dit d'où vient chaque ligne ; `validate="many_to_one"` vérifie qu'une clé n'apparaît qu'une fois à droite (sinon les lignes se dupliquent sans prévenir).

La table de référence ci-dessous décrit les entreprises. Elle contient volontairement **Airbus**, qui n'est pas dans nos données de cours. Exécute-la.

In [ ]:
infos = pd.DataFrame({
    "Ticker": ["AAPL", "MSFT", "MC.PA", "TTE.PA", "AIR.PA"],
    "nom": ["Apple", "Microsoft", "LVMH", "TotalEnergies", "Airbus"],
    "secteur": ["Technologie", "Technologie", "Luxe", "Énergie", "Aéronautique"],
    "pays": ["États-Unis", "États-Unis", "France", "France", "France"],
    "devise": ["USD", "USD", "EUR", "EUR", "EUR"],
})
infos

### Exercice 9.1 — Enrichir les cours

Ajoute à `long` les colonnes de `infos` (jointure sur `Ticker`, `how="left"`, avec `validate="many_to_one"`). Appelle le résultat `long_infos`.

Vérifie que le nombre de lignes n'a pas changé.

In [ ]:
# Ton code ici

### Exercice 9.2 — Les quatre types de jointure

Joins `long` et `infos` avec chacun des quatre `how`, et affiche le nombre de lignes obtenu. Puis, pour la jointure `"outer"`, ajoute `indicator=True` et compte les valeurs de la colonne `_merge`.

D'où vient la ligne en plus ?

In [ ]:
# Ton code ici

### Exercice 9.3 — Convertir en euros avec `merge`

Les cours d'Apple et de Microsoft sont en dollars. Pour comparer en euros, il faut le taux de change **du même jour**.

1. Crée `fx` : un tableau à deux colonnes, `Date` et `eurusd` (le `Close` de `fx_brut`).
2. Joins `long_infos` et `fx` sur la colonne `Date`, avec `how="left"`.
3. Combien de lignes n'ont pas trouvé de taux de change ? À quelles dates ?

`EURUSD=X` indique combien de dollars vaut un euro : un prix en dollars se convertit en euros en le **divisant** par ce taux.

In [ ]:
# Ton code ici

### Exercice 9.4 — `merge_asof` : la date la plus proche

`merge_asof` est un `merge` pour les séries temporelles : au lieu d'exiger une date **identique**, il prend pour chaque ligne la **dernière** valeur disponible à cette date ou avant (`direction="backward"`). C'est l'équivalent d'un `ffill` au moment de la jointure.

Les deux tableaux doivent être **triés** sur la clé.

1. Crée `long_eur` avec `pd.merge_asof` entre `long_infos` et `fx` (triés par `Date`).
2. Vérifie qu'il ne reste plus de taux manquant.
3. Ajoute la colonne `cloture_eur` : le cours divisé par `eurusd` pour les actions en dollars, inchangé pour les actions en euros.

💡 `np.where(condition, valeur_si_vrai, valeur_si_faux)`

In [ ]:
# Ton code ici

---
# Partie 10 — Calculs par action : `pivot`, `melt` et `groupby`

En format long, chaque calcul « dans le temps » doit être fait **action par action** : c'est le rôle de `groupby`. On retrouve l'équivalent des *window functions* SQL (`OVER (PARTITION BY ticker ORDER BY date)`).

| Besoin | Code |
|---|---|
| Format long → large | `long.pivot(index="Date", columns="nom", values="cloture_eur")` |
| Format large → long | `large.reset_index().melt(id_vars="Date", var_name="nom", value_name="cloture_eur")` |
| Rendement par action | `long.groupby("Ticker")["prix"].pct_change()` |
| Décalage par action | `long.groupby("Ticker")["prix"].shift(1)` |
| Une valeur de groupe recopiée sur chaque ligne | `long.groupby("Ticker")["Volume"].transform("mean")` |
| Fenêtre glissante par action | `long.groupby("Ticker")["prix"].transform(lambda s: s.rolling(20).mean())` |
| Classement dans chaque groupe | `long.groupby("Date")["rendement"].rank(ascending=False)` |

**`agg` ou `transform` ?** `agg` renvoie **une ligne par groupe** (la moyenne d'Apple) ; `transform` renvoie **une valeur par ligne d'origine** (la moyenne d'Apple, recopiée sur chaque ligne d'Apple), ce qui permet de comparer chaque ligne à son groupe.

### Exercice 10.1 — `pivot` et `melt`

1. Crée `prix_eur` : un tableau large des cours en euros, avec une colonne par entreprise (`nom`).
2. Remets-le au format long avec `melt` et compare le nombre de lignes avec `long_eur`. D'où vient la différence ?

In [ ]:
# Ton code ici

### Exercice 10.2 — Le piège du `shift` en format long

1. Trie `long_eur` par `Ticker` puis par `Date`, et renumérote les lignes (`reset_index(drop=True)`).
2. Calcule `rendement_faux` avec un simple `pct_change()` sur `cloture_eur`, puis `rendement` avec `groupby("Ticker")`.
3. Affiche la première ligne de chaque action (`groupby("Ticker").head(1)`) : que vaut `rendement_faux` à ces endroits ?

In [ ]:
# Ton code ici

### Exercice 10.3 — `transform` : comparer chaque séance à son action

1. Ajoute `volume_moyen` : le volume moyen de l'action, recopié sur chaque ligne.
2. Ajoute `volume_relatif` : le volume de la séance divisé par ce volume moyen.
3. Affiche les 5 séances au volume le plus exceptionnel, toutes actions confondues.
4. Pour comparer, affiche le volume moyen par action calculé avec `agg` : combien de lignes ?

Pourquoi ne peut-on pas simplement comparer les volumes bruts d'une action à l'autre ?

In [ ]:
# Ton code ici

### Exercice 10.4 — Une fenêtre glissante par action

Ajoute `mm_20_eur` : la moyenne mobile sur 20 séances du cours en euros, calculée action par action. Vérifie qu'il y a exactement 19 valeurs manquantes par action.

In [ ]:
# Ton code ici

### Exercice 10.5 — `rank` : le classement mensuel

1. À partir de `prix_eur`, calcule les rendements mensuels en euros.
2. Classe les quatre actions chaque mois, de la meilleure (1) à la moins bonne (4), avec `rank(axis=1, ascending=False)`.
3. Combien de fois chaque action a-t-elle terminé première ?

💡 `prix_eur` contient des trous les jours fériés : comble-les avec `ffill()` avant de rééchantillonner.

In [ ]:
# Ton code ici

### Exercice 10.6 — Par secteur

Grâce au `merge` avec `infos`, on peut raisonner par secteur. Calcule, pour chaque secteur, le rendement journalier moyen (en %), la volatilité annualisée (en %) et le nombre d'observations.

In [ ]:
# Ton code ici

---
# Partie 11 — Découper en classes : `qcut` et `cut`

| | `pd.qcut(s, q=5)` | `pd.cut(s, bins=[…])` |
|---|---|---|
| Découpe selon | les **quantiles** | des **bornes fixes** que tu choisis |
| Taille des classes | **égale** (autant de lignes dans chaque classe) | variable |
| Usage typique | « les 20 % de séances aux plus forts volumes » | « les baisses de plus de 2 % » |

### Exercice 11.1 — Les quintiles de volume

Les marchés bougent-ils davantage les jours où l'on échange beaucoup ?

1. Dans `aapl`, crée `quintile_volume` avec `pd.qcut` : 5 classes nommées de `"Q1 (faible)"` à `"Q5 (fort)"`.
2. Vérifie que chaque classe contient le même nombre de séances.
3. Calcule, pour chaque quintile, la variation moyenne **en valeur absolue** (`.abs()`) du rendement, en %.

💡 Pour un `groupby` sur une colonne de catégories, ajoute `observed=True`.

In [ ]:
# Ton code ici

### Exercice 11.2 — `cut` contre `qcut`

1. Classe les rendements d'Apple avec `pd.cut` dans 5 catégories à bornes fixes : forte baisse (moins de −2 %), baisse (de −2 % à −0,5 %), stable (de −0,5 % à +0,5 %), hausse (de +0,5 % à +2 %), forte hausse (plus de +2 %). Compte les séances de chaque catégorie.
2. Découpe maintenant les rendements en 5 classes avec `pd.qcut` et affiche les bornes obtenues (`.cat.categories`). Compare.

💡 Pour des bornes ouvertes, utilise `-np.inf` et `np.inf`.

In [ ]:
# Ton code ici

---
# Partie 12 — Les corrélations

La **corrélation** mesure à quel point deux séries évoluent ensemble, de −1 (en sens opposé) à +1 (exactement ensemble). En gestion de portefeuille, elle mesure la **diversification** : associer des actions peu corrélées réduit le risque global.

On calcule toujours la corrélation sur les **rendements**, jamais sur les prix : deux actions qui montent toutes les deux sur la période auraient des prix très « corrélés », même si leurs mouvements quotidiens n'ont rien à voir.

### Exercice 12.1 — La matrice de corrélation

Calcule la matrice de corrélation de `rendements` et affiche-la sous forme de carte de chaleur avec `px.imshow` (`text_auto=".2f"`, échelle de −1 à 1).

In [ ]:
# Ton code ici

### Exercice 12.2 — Les paires les plus corrélées

Transforme la matrice en une liste de paires triées de la plus corrélée à la moins corrélée, **sans doublons** (A-B et B-A) ni diagonale (A-A).

💡 `np.triu(np.ones(matrice.shape), k=1).astype(bool)` est un masque du triangle supérieur, sans la diagonale. `matrice.where(masque)` ne garde que ces cases, puis `.stack()` les met en liste, et `.dropna()` retire les cases écartées.

In [ ]:
# Ton code ici

---
# Partie 13 — Projet : tester une stratégie de moyennes mobiles

On teste une règle classique sur Apple, le **croisement de moyennes mobiles** :

- quand la moyenne sur 20 séances est **au-dessus** de celle sur 50 séances (tendance haussière), on détient l'action ;
- sinon, on reste en liquidités (rendement nul).

On la compare à la stratégie la plus simple : acheter au début et conserver (*buy and hold*).

### Exercice 13.1 — Le signal

Dans `aapl`, crée `signal` : 1 si `mm_20` est au-dessus de `mm_50`, 0 sinon. Quelle part du temps la stratégie est-elle investie ?

In [ ]:
# Ton code ici

### Exercice 13.2 — La performance de la stratégie

Le signal est calculé avec le cours de **clôture** : on ne peut donc agir qu'à la séance **suivante**. Le rendement de la stratégie un jour donné est le signal de la **veille** multiplié par le rendement du jour.

1. Crée `rendement_strategie` avec `shift(1)`.
2. Calcule la performance cumulée (base 100) de la stratégie et de l'achat-conservation, et trace-les sur le même graphique.

In [ ]:
# Ton code ici

### Exercice 13.3 — Le biais d'anticipation

1. Recalcule la stratégie de croisement **sans** le `shift(1)`, comme si l'on pouvait décider le matin en connaissant déjà la clôture du soir. Compare la performance finale des deux versions.
2. L'écart est faible, car les moyennes mobiles changent lentement : l'erreur passe facilement inaperçue. Pour la rendre évidente, teste une règle volontairement absurde : « détenir l'action les jours où elle monte » (`signal_jour = rendement > 0`), avec et sans `shift(1)`.

Pourquoi les versions sans `shift` sont-elles trompeuses ?

In [ ]:
# Ton code ici

### Exercice 13.4 — Le bilan

Écris une fonction `bilan(r)` qui reçoit une série de rendements journaliers et renvoie un dictionnaire avec :

- la performance totale (%) ;
- la volatilité annualisée (%) ;
- la pire baisse depuis un plus haut (%).

Applique-la aux deux stratégies et présente le résultat dans un tableau. La stratégie bat-elle l'achat-conservation ? Sur quel critère ?

In [ ]:
# Ton code ici

---
# Bonus — Écrire en chaîne avec `assign` et `pipe`

Au lieu de créer les colonnes une par une, on peut enchaîner les transformations dans une seule expression, entre parenthèses :

- `assign(nom=lambda d: …)` crée une colonne (le `d` désigne le tableau à ce stade de la chaîne) ;
- `pipe(fonction)` passe tout le tableau à une fonction, pour insérer ses propres étapes dans la chaîne.

Le code se lit alors de haut en bas, comme une recette.

### Exercice B.1 — Les indicateurs de Microsoft en une seule chaîne

En partant de `brut.xs("MSFT", axis=1, level="Ticker")`, construis `msft` en **une seule expression** qui :

1. supprime les lignes vides ;
2. ajoute `rendement`, `mm_20`, `mm_50` et `signal` avec `assign` ;
3. ne garde que les séances depuis janvier 2026.

In [ ]:
# Ton code ici

### Exercice B.2 — Le MACD avec `pipe`

Le **MACD** est un indicateur très utilisé, construit avec des moyennes exponentielles :

- `macd` = moyenne exponentielle sur 12 séances − moyenne exponentielle sur 26 séances ;
- `ligne_signal` = moyenne exponentielle sur 9 séances du `macd`.

Écris une fonction `ajouter_macd(d)` qui reçoit un tableau contenant `Close` et renvoie ce tableau avec ces deux colonnes, puis applique-la à Apple avec `pipe`. Trace `macd` et `ligne_signal` depuis janvier 2026.

In [ ]:
# Ton code ici

---
## 🎉 Terminé !

Tu as parcouru toute la chaîne d'une analyse financière avec pandas : récupérer les données, les nettoyer, calculer rendements et risques, changer de fréquence, combiner plusieurs sources avec `concat` et `merge`, raisonner par action et par secteur, et tester une stratégie sans tricher avec le futur.

**Pour aller plus loin**

- Ajoute une action de ton choix dans `ACTIONS` (par exemple `AIR.PA`) et relance tout le notebook. Le `merge` de la partie 9 trouvera alors une correspondance pour Airbus.
- Teste d'autres longueurs de moyennes mobiles (10/30, 50/200) : la conclusion de la partie 13 tient-elle ?
- Construis un portefeuille équipondéré des quatre actions (la moyenne des rendements en euros chaque jour) et compare son drawdown à celui de chaque action : c'est l'effet de la diversification.